In [ ]:
import numpy as np
import pandas as pd
import yaml
import re
import json

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=True)

In [ ]:
import os
os.chdir('../../')

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

## Import data

### Check US firm data

In [ ]:
df_pubs = pd.read_csv(dataset_config['path_processed'] + 'WOS/WOS_USfirm_publication.csv')
df_pubs

In [ ]:
df_agg = (
    df_pubs.groupby("affiliationame", as_index=False)
      .agg(total_papers=("num_papers", "sum"))
)
df_agg

### Firm name link

In [ ]:
import json

file_path = dataset_config['path_processed'] + 'WOS/05_affiliation_clusters_US.json'

with open(file_path, 'r', encoding='utf-8') as f:
    cluster_data = json.load(f)

mapping = {}
total = 0

for cluster in cluster_data:
    if len(cluster) > 1:  # skip singletons
        shortest = min(cluster, key=len)  # find shortest string
        mapping[shortest] = []
        for item in cluster:
            mapping[shortest].append(item)
            total += 1

total

In [ ]:
check_filtered = df_agg[df_agg.total_papers <= 1]
check_filtered

In [ ]:
df_mappings = (
    pd.DataFrame([
        {"affiliationame": key, "canonical_affiliation": raw}
        for key, values in mapping.items()
        for raw in values
    ])
)
df_mappings

In [ ]:
repeated_entry = df_mappings.merge(check_filtered)
repeated_entry

In [ ]:
df_canonical = pd.read_parquet(dataset_config['path_processed'] + 'WOS/04_canonical_firms_US.parquet')
df_canonical

In [ ]:
df_firms_rep = repeated_entry[['canonical_affiliation']].merge(df_canonical[['organization', 'canonical_affiliation']])
df_firms_rep

In [ ]:
df_firms = pd.read_csv(dataset_config['path_processed'] + 'WOS/POST00_US_firms_filtered.csv')
df_firms

In [ ]:
name_link = df_firms.merge( pd.concat([df_canonical[['organization', 'canonical_affiliation']], df_firms_rep])).rename(columns={'canonical_affiliation': 'affiliationame',
                                                                                                    'organization': 'organizations'}).merge(check_filtered)
name_link

## Merge

In [ ]:
df_wos = pd.read_csv(dataset_config['path_processed'] + 'WOS/WOS_2010_2023.csv')
df_wos

In [ ]:
df_low_freq = name_link.merge(df_wos)
df_low_freq

In [ ]:
df_low_freq = df_low_freq.drop_duplicates(subset=['wosid']).drop(columns=['researcher_id', 'seq_no', 'reprint'])
df_low_freq

In [ ]:
wos_paper_info = pd.read_parquet(dataset_config['path_processed'] + 'WOS/WOS_paper_level.parquet').drop_duplicates()
wos_paper_info.rename(columns={'pub year': 'year'}, inplace=True)
wos_paper_info

In [ ]:
df_wos_title = df_low_freq.merge(wos_paper_info)
df_wos_title

In [ ]:
df_final = df_wos_title[df_wos_title.year <= 2022]
df_final

In [ ]:
df_final.to_csv(dataset_config['path_processed'] + 'WOS/CHECK01_WOS_low_freq.csv', index=False)